# Protein structure analysis pipeline

Build a mini protein's backbone from φ/ψ angles using internal-coordinate geometry, write a PDB file, recover dihedrals, and analyze the structure.

In [ ]:
import numpy as np

B_NCA, B_CAC, B_CN = 1.458, 1.525, 1.329
A_NCAC, A_CACN, A_CNCA = 111.0, 116.2, 121.7

def place(a, b, c, bond, angle, dih):
    ba, bc = a - b, c - b
    n = np.cross(bc, ba); n /= np.linalg.norm(n)
    m = np.cross(n, bc); m /= np.linalg.norm(m)
    ar, dr = np.radians(angle), np.radians(dih)
    u = bc / np.linalg.norm(bc)
    return c + bond * (u*np.cos(ar) + m*np.sin(ar)*np.cos(dr) + n*np.sin(ar)*np.sin(dr))

def dihedral(p0, p1, p2, p3):
    b0 = -(p1 - p0); b1 = p2 - p1; b2 = p3 - p2
    b1 /= np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1)*b1; w = b2 - np.dot(b2, b1)*b1
    return np.degrees(np.arctan2(np.dot(np.cross(b1, v), w), np.dot(v, w)))

In [ ]:
rng = np.random.default_rng(11)
seq = 'MKTAYIAKQRQISFVKSHFSRQDILDLWQKAHALEVNEKQLAARLKELGYVESGTLEDVDE'[:60]
segs = [(0,20,-57,-47,6.0),(20,28,-65,140,25.0),(28,45,-120,130,12.0),(45,60,-57,-47,6.0)]
phis, psis = [], []
for s,e,p0,q0,nz in segs:
    for _ in range(s,e):
        phis.append(p0 + rng.normal(0,nz)); psis.append(q0 + rng.normal(0,nz))
phis[0], psis[-1] = -60.0, 130.0
n = len(phis)
atoms = {'N':[None]*n, 'CA':[None]*n, 'C':[None]*n, 'O':[None]*n}
atoms['N'][0] = np.zeros(3); atoms['CA'][0] = np.array([B_NCA, 0, 0])
atoms['C'][0] = atoms['CA'][0] + np.array([B_CAC*np.cos(np.radians(69)), B_CAC*np.sin(np.radians(69)), 0])
for i in range(n-1):
    Ni, CAi, Ci = atoms['N'][i], atoms['CA'][i], atoms['C'][i]
    Nn = place(Ni, CAi, Ci, B_CN, A_CACN, psis[i])
    CAn = place(CAi, Ci, Nn, B_NCA, A_CNCA, 180.0)
    Cn = place(Ci, Nn, CAn, B_CAC, A_NCAC, phis[i+1])
    atoms['N'][i+1], atoms['CA'][i+1], atoms['C'][i+1] = Nn, CAn, Cn
rphis = [dihedral(atoms['C'][i-1] if i else atoms['CA'][i], atoms['N'][i], atoms['CA'][i], atoms['C'][i]) for i in range(n)]
rpsis = [dihedral(atoms['N'][i], atoms['CA'][i], atoms['C'][i], atoms['N'][i+1] if i<n-1 else atoms['CA'][i]) for i in range(n)]

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.2, 4.8))
ax.scatter(rphis, rpsis, s=22, c='#4f8cff')
ax.axhline(0, color='#8b97a5', lw=0.6, ls=':'); ax.axvline(0, color='#8b97a5', lw=0.6, ls=':')
ax.add_patch(plt.Rectangle((-90,-90),60,60,facecolor='#35c4b6',alpha=0.12))
ax.add_patch(plt.Rectangle((-170,90),130,90,facecolor='#d9a441',alpha=0.10))
ax.set_xlim(-180,180); ax.set_ylim(-180,180)
ax.set_xlabel('phi (deg)'); ax.set_ylabel('psi (deg)')
ax.set_title('Ramachandran plot')

In [ ]:
from collections import Counter
def ss(p,q):
    if -90 <= p <= -30 and -90 <= q <= -30: return 'alpha'
    if -170 <= p <= -40 and 90 <= q <= 180: return 'beta'
    return 'coil'
ssc = Counter(ss(p,q) for p,q in zip(rphis,rpsis))
comp = Counter(seq)
print('secondary structure:', dict(ssc))
print('most common residues :', comp.most_common(5))
print(f'residues: {n}, alpha={ssc["alpha"]}, beta={ssc["beta"]}, coil={ssc["coil"]}')